# Mass-Spring System (No Damping, Didactic Version)

Workflow:
1. Solve with Euler
2. Show raw numerical output
3. Show spring animation + position plot
4. Repeat with RK4


Equation of motion: `x'' + (k/m)*x = 0`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
try:
    from IPython.display import HTML
except Exception:
    HTML = None

# BLOCK 1: PARAMETERS
m = 1.0
k = 1.0
x0 = 1.0
v0 = 0.0
dt = 0.01
t_final = 20.0
t = np.arange(0.0, t_final + dt, dt)
N = len(t)

# BLOCK 2: EULER METHOD
def simular_euler():
    x = np.zeros(N)
    v = np.zeros(N)
    x[0] = x0
    v[0] = v0
    for i in range(N - 1):
        a = -(k / m) * x[i]
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i] * dt
    return x, v

# BLOCK 3: RK4 METHOD
def derivadas(x, v):
    return v, -(k / m) * x

def simular_rk4():
    x = np.zeros(N)
    v = np.zeros(N)
    x[0] = x0
    v[0] = v0
    for i in range(N - 1):
        xi, vi = x[i], v[i]
        k1_x, k1_v = derivadas(xi, vi)
        k2_x, k2_v = derivadas(xi + 0.5 * dt * k1_x, vi + 0.5 * dt * k1_v)
        k3_x, k3_v = derivadas(xi + 0.5 * dt * k2_x, vi + 0.5 * dt * k2_v)
        k4_x, k4_v = derivadas(xi + dt * k3_x, vi + dt * k3_v)
        x[i + 1] = xi + (dt / 6.0) * (k1_x + 2 * k2_x + 2 * k3_x + k4_x)
        v[i + 1] = vi + (dt / 6.0) * (k1_v + 2 * k2_v + 2 * k3_v + k4_v)
    return x, v

# BLOCK 4: HELPER FUNCTIONS
def show_raw_output(method_name, x, v):
    print(f"\nRaw output - {method_name}")
    print(" i   t(s)      x(m)        v(m/s)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {x[i]:10.6f}   {v[i]:10.6f}")

def draw_spring(x_mass, x_wall=-1.2, turns=12, amp=0.08, points=220):
    x_end = x_mass - 0.08
    if x_end <= x_wall + 0.05:
        x_end = x_wall + 0.05
    xs = np.linspace(x_wall, x_end, points)
    phase = np.linspace(0.0, 2.0 * np.pi * turns, points)
    ys = amp * np.sin(phase)
    ys[0] = 0.0
    ys[-1] = 0.0
    return xs, ys

def animate_spring(method_name, x):
    fig, (ax_spring, ax_xt) = plt.subplots(2, 1, figsize=(8, 7), gridspec_kw={'height_ratios': [1.3, 1.0]})

    ax_spring.set_title(f'{method_name} - Spring motion')
    ax_spring.set_xlabel('x (m)')
    ax_spring.set_ylabel('y')
    xmin = min(-1.3, np.min(x) - 0.3)
    xmax = max(1.3, np.max(x) + 0.3)
    ax_spring.set_xlim(xmin, xmax)
    ax_spring.set_ylim(-0.4, 0.4)
    ax_spring.grid(alpha=0.3)
    ax_spring.axvline(-1.2, color='gray', lw=3)

    spring_line, = ax_spring.plot([], [], color='tab:blue', lw=2)
    mass_point, = ax_spring.plot([], [], 'o', color='tab:red', ms=14)

    ax_xt.set_title(f'{method_name} - Position x(t)')
    ax_xt.set_xlabel('time (s)')
    ax_xt.set_ylabel('x (m)')
    ax_xt.set_xlim(0.0, t_final)
    margin = 0.1 * max(abs(np.min(x)), abs(np.max(x)), 1e-6)
    ax_xt.set_ylim(np.min(x) - margin, np.max(x) + margin)
    ax_xt.grid(alpha=0.3)
    x_line, = ax_xt.plot([], [], color='tab:orange', lw=2)

    fig.subplots_adjust(bottom=0.14, hspace=0.4)
    fig.text(0.02, 0.04, "ODE: x'' + (k/m)*x = 0", fontsize=10)

    def init():
        spring_line.set_data([], [])
        mass_point.set_data([], [])
        x_line.set_data([], [])
        return spring_line, mass_point, x_line

    def update(i):
        xs, ys = draw_spring(x[i])
        spring_line.set_data(xs, ys)
        mass_point.set_data([x[i]], [0.0])
        x_line.set_data(t[:i+1], x[:i+1])
        return spring_line, mass_point, x_line

    ani = FuncAnimation(fig, update, frames=N, init_func=init, interval=12, blit=False)
    plt.close(fig)
    if HTML is None:
        return ani
    return HTML(ani.to_jshtml())



In [ ]:
# BLOCK 5: RUN EULER
x_euler, v_euler = simular_euler()
show_raw_output('Euler', x_euler, v_euler)
animate_spring('Euler', x_euler)


In [ ]:
# BLOCK 6: RUN RK4
x_rk4, v_rk4 = simular_rk4()
show_raw_output('Runge-Kutta 4', x_rk4, v_rk4)
animate_spring('Runge-Kutta 4', x_rk4)
